In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


In [ ]:
def flatten_data(folder_path):
    data_dir = Path(folder_path)
    final_data = []

    
    for i in tqdm(list(data_dir.glob('*.json'))):
        with i.open('r') as f:
            match_data = json.load(f)

        
        info = match_data.get('info', {})
        if info.get('gender') != 'male' or info.get('match_type') != 'T20':
            continue

       
        venue = info.get('venue', 'Unknown')
        
       
        outcome = info.get('outcome', {})
        winner = outcome.get('winner')

        toss = info.get('toss', {})
        toss_winner = toss.get('winner')
        

        match_date = info.get('dates', ['1970-01-01'])[0]  
        match_id = i.stem

        
        if len(match_data.get('innings', [])) < 2:
            continue
            
        second_innings = match_data['innings'][1]
        
       
        target = second_innings.get('target', {}).get('runs', 0)
        if target == 0: 
            continue 

        batting_team = second_innings.get('team')
        
       
        teams = info.get('teams', [])
        bowling_team_list = [t for t in teams if t != batting_team]
        if not bowling_team_list:
            continue
        bowling_team = bowling_team_list[0]

        
        is_chaser_winner = 1 if winner == batting_team else 0

        chasing_team_toss_winner=1 if toss_winner==batting_team else 0


        current_score = 0
        wickets_lost = 0

        
        for over in second_innings.get('overs', []):
            over_num = over.get('over', 0)
            
            for ball_index, delivery in enumerate(over.get('deliveries', [])):
                
                runs = delivery.get('runs', {}).get('total', 0)
                current_score += runs
                
                if 'wickets' in delivery:
                    wickets_lost += len(delivery['wickets'])

                
                runs_left = target - current_score
                balls_delivered = (over_num * 6) + (ball_index + 1)
                balls_left = 120 - balls_delivered
                wickets_left = 10 - wickets_lost
                
                
                if balls_left < 0: 
                    balls_left = 0

                
                crr = (current_score * 6) / balls_delivered if balls_delivered > 0 else 0
                rrr = (runs_left * 6) / balls_left if balls_left > 0 else 0

                striker = delivery.get('batter')
                non_striker = delivery.get('non_striker')
                current_bowler = delivery.get('bowler')

                final_data.append({
                    'batting_team': batting_team,
                    'bowling_team': bowling_team,
                    'venue': venue,
                    'runs_left': runs_left,
                    'balls_left': balls_left,
                    'wickets_left': wickets_left,
                    'target_runs': target,
                    'crr': crr,
                    'rrr': rrr,
                    'result': is_chaser_winner,
                    'striker': striker,              
                    'non_striker': non_striker,      
                    'bowler': current_bowler,
                    'chasing_team_toss_winner':chasing_team_toss_winner,
                    "match_id":match_id,
                    'date':match_date
                })

    return pd.DataFrame(final_data)

In [4]:
df=flatten_data("D:\Semester 4\AI\Lab\Project\\t20s_json")

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\ammar\AppData\Local\Temp\ipykernel_11820\562114739.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  df=flatten_data("D:\Semester 4\AI\Lab\Project\\t20s_json")
  0%|          | 0/5211 [00:00<?, ?it/s]

100%|██████████| 5211/5211 [00:03<00:00, 1319.19it/s]


In [5]:
#removing the rows where the match is already won
df=df[df['runs_left']>=0]

#removing rain affected or unrealistic matches
df=df[df['target_runs']>=50]

#this will avoid infinite rrr error
df=df[df['balls_left']>0]

In [6]:
import pandas as pd


match_df = df[['match_id', 'date', 'batting_team', 'bowling_team', 'result']].drop_duplicates()


match_df['matchup'] = match_df.apply(lambda x: tuple(sorted([x['batting_team'], x['bowling_team']])), axis=1)


match_df = match_df.sort_values('date')

def get_last_5_wins(row, all_matches):
    
    past_matches = all_matches[
        (all_matches['matchup'] == row['matchup']) & 
        (all_matches['date'] < row['date'])
    ].tail(5)
    
    if past_matches.empty:
        return 0 
        
   
    wins = 0
    for _, match in past_matches.iterrows():
      
        if match['batting_team'] == row['batting_team'] and match['result'] == 1:
            wins += 1
        
        elif match['bowling_team'] == row['batting_team'] and match['result'] == 0:
            wins += 1
            
    return wins



match_df['chasing_team_last_5_wins'] = match_df.apply(lambda x: get_last_5_wins(x, match_df), axis=1)


df = df.merge(match_df[['match_id', 'chasing_team_last_5_wins']], on='match_id', how='left')


In [7]:
df.describe()

,runs_left,balls_left,wickets_left,target_runs,crr,rrr,result,chasing_team_toss_winner,chasing_team_last_5_wins
count,336861.000000,336861.000000,336861.000000,336861.000000,336861.000000,336861.000000,336861.000000,336861.000000,336861.000000
mean,91.743491,65.916693,7.216531,155.548763,7.003777,10.800250,0.454837,0.540962,1.281909
std,50.888223,32.510677,2.348596,38.507807,2.617950,19.160285,0.497957,0.498320,1.422939
min,0.000000,1.000000,0.000000,50.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,52.000000,39.000000,6.000000,130.000000,5.454545,6.100840,0.000000,0.000000,0.000000
50%,89.000000,68.000000,8.000000,155.000000,6.900000,8.387755,0.000000,1.000000,1.000000
75%,127.000000,94.000000,9.000000,180.000000,8.432432,11.025000,1.000000,1.000000,2.000000
max,345.000000,119.000000,10.000000,345.000000,36.000000,1308.000000,1.000000,1.000000,5.000000


In [10]:
#converting to csv file
df.to_csv('processed_data.csv',index=False)

In [9]:
df.head()

,batting_team,bowling_team,venue,runs_left,balls_left,wickets_left,target_runs,crr,rrr,result,striker,non_striker,bowler,chasing_team_toss_winner,match_id,date,chasing_team_last_5_wins
0,Sri Lanka,Australia,Melbourne Cricket Ground,168,119,10,169,6.0,8.470588,1,N Dickwella,WU Tharanga,PJ Cummins,1,1001349,2017-02-17,3
1,Sri Lanka,Australia,Melbourne Cricket Ground,167,118,10,169,6.0,8.491525,1,WU Tharanga,N Dickwella,PJ Cummins,1,1001349,2017-02-17,3
2,Sri Lanka,Australia,Melbourne Cricket Ground,167,117,10,169,4.0,8.564103,1,N Dickwella,WU Tharanga,PJ Cummins,1,1001349,2017-02-17,3
3,Sri Lanka,Australia,Melbourne Cricket Ground,167,116,10,169,3.0,8.637931,1,N Dickwella,WU Tharanga,PJ Cummins,1,1001349,2017-02-17,3
4,Sri Lanka,Australia,Melbourne Cricket Ground,164,115,10,169,6.0,8.556522,1,N Dickwella,WU Tharanga,PJ Cummins,1,1001349,2017-02-17,3
